# Customer Churn Exploratory Data Analysis (EDA)

This notebook explores the IBM Telco Customer Churn dataset to identify key drivers of churn and prepare the data for predictive modeling.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## 1. Load and Clean Dataset

In [ ]:
data_path = os.path.join('..', 'data', 'customer_churn.csv')
df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape}")
df.head()

In [ ]:
# Data Info and Missing Values
df.info()

In [ ]:
# Convert TotalCharges to numeric, replacing spaces with NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].replace(' ', np.nan), errors='coerce')
print(f"Missing TotalCharges: {df['TotalCharges'].isna().sum()}")

# Drop missing values
df = df.dropna(subset=['TotalCharges'])
print(f"Shape after dropping missing values: {df.shape}")

In [ ]:
# Check for duplicates
print(f"Duplicate rows: {df.duplicated().sum()}")

## 2. Univariate Analysis

### Target Variable: Churn Distribution

In [ ]:
churn_counts = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True) * 100

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
sns.countplot(data=df, x='Churn', ax=ax[0], palette='Blues_r')
ax[0].set_title('Churn Class Count')
for p in ax[0].patches:
    ax[0].annotate(f'{int(p.get_height())}', (p.get_x() + 0.35, p.get_height() + 50))

ax[1].pie(churn_pct, labels=churn_pct.index, autopct='%1.1f%%', colors=['#3498db', '#e74c3c'], startangle=90, explode=[0, 0.1])
ax[1].set_title('Churn Class Percentage')
plt.suptitle('Churn Target Distribution (Highly Imbalanced)')
plt.show()

### Numerical Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.histplot(data=df, x='tenure', kde=True, ax=axes[0], color='skyblue')
axes[0].set_title('Tenure Distribution')

sns.histplot(data=df, x='MonthlyCharges', kde=True, ax=axes[1], color='salmon')
axes[1].set_title('Monthly Charges Distribution')

sns.histplot(data=df, x='TotalCharges', kde=True, ax=axes[2], color='lightgreen')
axes[2].set_title('Total Charges Distribution')

plt.tight_layout()
plt.show()

## 3. Bivariate Analysis: Answering Key Questions

### Q1. Which customers churn the most? (Demographics & Service Options)
Let's look at gender, partner, senior citizen, internet service, payment method, contract types.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Gender vs Churn
sns.countplot(data=df, x='gender', hue='Churn', ax=axes[0, 0], palette='pastel')
axes[0, 0].set_title('Churn Rate by Gender')

# Senior Citizen vs Churn
sns.countplot(data=df, x='SeniorCitizen', hue='Churn', ax=axes[0, 1], palette='pastel')
axes[0, 1].set_title('Churn Rate by Senior Citizen')

# Partner vs Churn
sns.countplot(data=df, x='Partner', hue='Churn', ax=axes[1, 0], palette='pastel')
axes[1, 0].set_title('Churn Rate by Partner')

# Internet Service vs Churn
sns.countplot(data=df, x='InternetService', hue='Churn', ax=axes[1, 1], palette='pastel')
axes[1, 1].set_title('Churn Rate by Internet Service')

plt.tight_layout()
plt.show()

**Insight**: 
- Gender has almost no impact on Churn (male and female rates are identical).
- Senior citizens have a significantly higher rate of churn.
- Customers without a partner churn more.
- Customers with Fiber optic internet service churn at a much higher rate than DSL or no internet service customers.

### Q2. Which contract churns more?

In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='Contract', hue='Churn', palette='Set2')
plt.title('Churn Count by Contract Type')
plt.show()

# Percentage breakdown
df.groupby('Contract')['Churn'].value_counts(normalize=True).unstack() * 100

**Insight**: 
- Month-to-month contracts have an extremely high churn rate (~42.7%), whereas one-year (~11.3%) and two-year (~2.8%) contracts have very low churn. Locking customers into contracts is a key retention driver.

### Q3. Does tenure affect churn?

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

# Boxplot
sns.boxplot(data=df, x='Churn', y='tenure', ax=ax[0], palette='Set1')
ax[0].set_title('Tenure Distribution vs Churn')

# KDE plot
sns.kdeplot(data=df, x='tenure', hue='Churn', fill=True, common_norm=False, alpha=0.5, ax=ax[1], palette='Set1')
ax[1].set_title('Tenure Density vs Churn')

plt.show()

**Insight**:
- Yes, tenure heavily affects churn. Churned customers have a much lower median tenure (around 10 months) compared to retained customers (around 38 months). Most churn occurs in the first 1-12 months.

### Q4. Does monthly charge matter?

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

# Boxplot
sns.boxplot(data=df, x='Churn', y='MonthlyCharges', ax=ax[0], palette='Set3')
ax[0].set_title('Monthly Charges vs Churn')

# KDE plot
sns.kdeplot(data=df, x='MonthlyCharges', hue='Churn', fill=True, common_norm=False, alpha=0.5, ax=ax[1], palette='Set3')
ax[1].set_title('Monthly Charges Density vs Churn')

plt.show()

**Insight**:
- Churned customers generally have higher monthly charges (median around $80) compared to retained customers (median around $65). The density plot shows a spike of churned customers at high charges ($70 - $100).

## 4. Correlation Analysis

In [ ]:
# Convert object variables to categories/codes to compute correlation
df_corr = df.copy()
for col in df_corr.select_dtypes(include=['object']).columns:
    df_corr[col] = df_corr[col].astype('category').cat.codes

plt.figure(figsize=(12, 10))
sns.heatmap(df_corr.corr(), annot=False, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix of All Encoded Variables')
plt.show()